<a href="https://colab.research.google.com/github/VinayBR03/bert-domestic-violence-risk-detector/blob/main/Domestic_Violence_Risk_Detector.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install speechrecognition
!apt-get update && apt-get install -y portaudio19-dev
!pip install PyAudio --no-cache-dir

Hit:1 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease
Hit:2 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease
Hit:3 http://archive.ubuntu.com/ubuntu jammy InRelease
Hit:4 http://security.ubuntu.com/ubuntu jammy-security InRelease
Hit:5 http://archive.ubuntu.com/ubuntu jammy-updates InRelease
Hit:6 http://archive.ubuntu.com/ubuntu jammy-backports InRelease
Hit:7 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:8 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Hit:9 https://r2u.stat.illinois.edu/ubuntu jammy InRelease
Hit:10 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Reading package lists... Done
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading

In [ ]:
!pip install pydub

In [ ]:
import torch
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from transformers import BertTokenizer, BertForSequenceClassification, AdamW
from torch.utils.data import DataLoader, Dataset
from tqdm import tqdm
import speech_recognition as sr
import io  # Import the io module for BytesIO
from IPython.display import display, Javascript
from google.colab import output,files
from base64 import b64decode
from pydub import AudioSegment


# Load pre-trained BERT model and tokenizer
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
model = BertForSequenceClassification.from_pretrained('bert-base-uncased', num_labels=2)

# Custom Dataset class for text data
class TextDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=500):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = self.texts[idx]
        label = self.labels[idx]
        encoding = self.tokenizer.encode_plus(
            text,
            add_special_tokens=True,
            max_length=self.max_len,
            return_token_type_ids=False,
            padding='max_length',
            truncation=True,
            return_attention_mask=True,
            return_tensors='pt',
        )
        return {
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten(),
            'label': torch.tensor(label, dtype=torch.long)
        }

# Function to train the BERT model
def train_model(train_loader, val_loader, epochs=4, lr=2e-5):
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model.to(device)
    optimizer = AdamW(model.parameters(), lr=lr)

    for epoch in range(epochs):
        print(f'Epoch {epoch + 1}/{epochs}')
        model.train()
        train_loss = 0
        for batch in tqdm(train_loader):
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['label'].to(device)

            optimizer.zero_grad()
            outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
            loss = outputs.loss
            train_loss += loss.item()

            loss.backward()
            optimizer.step()

        print(f'Train Loss: {train_loss / len(train_loader)}')

        # Validation
        model.eval()
        val_preds, val_labels = [], []
        with torch.no_grad():
            for batch in val_loader:
                input_ids = batch['input_ids'].to(device)
                attention_mask = batch['attention_mask'].to(device)
                labels = batch['label'].to(device)

                outputs = model(input_ids=input_ids, attention_mask=attention_mask)
                preds = torch.argmax(outputs.logits, dim=1)
                val_preds.extend(preds.cpu().numpy())
                val_labels.extend(labels.cpu().numpy())

        val_accuracy = accuracy_score(val_labels, val_preds)
        print(f'Validation Accuracy: {val_accuracy*100:.4f}')

# Function to analyze user input text
def analyze_user_input(text, model, tokenizer):
    inputs = tokenizer(text, return_tensors='pt', padding=True, truncation=True)

    # Move input tensors to the same device as the model
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    inputs = {k: v.to(device) for k, v in inputs.items()}

    model.eval()
    with torch.no_grad():
        outputs = model(**inputs)
        logits = outputs.logits
        probabilities = torch.softmax(logits, dim=1)
    return probabilities[0][1].item()  # Probability of high risk

# Load and preprocess training data
train_df = pd.read_csv('domestic_violence_data.csv')
texts = train_df['Text'].tolist()
labels = train_df['Label'].tolist()
train_texts, val_texts, train_labels, val_labels = train_test_split(texts, labels, test_size=0.2, random_state=42)

# Load and preprocess testing data
test_df = pd.read_csv('text_test.csv')
test_texts = test_df['text'].tolist()
test_labels = test_df['label'].tolist()

# Create datasets and data loaders
train_dataset = TextDataset(train_texts, train_labels, tokenizer)
val_dataset = TextDataset(val_texts, val_labels, tokenizer)
test_dataset = TextDataset(test_texts, test_labels, tokenizer)
train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=16, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=16, shuffle=False)

# Train the model
train_model(train_loader, val_loader)

# Function to get user input (text or audio)
def get_user_input():
    choice = input("Enter '1' for text input, '2' to record audio, or '3' to upload an MP3 file: ")

    if choice == '1':
        return input("Enter text to analyze: ")

    elif choice == '2':
        # Record audio using JavaScript
        RECORD = """
        const sleep  = time => new Promise(resolve => setTimeout(resolve, time))
        const b2text = blob => new Promise(resolve => {
          const reader = new FileReader()
          reader.onloadend = e => resolve(e.srcElement.result)
          reader.readAsDataURL(blob)
        })
        var record = time => new Promise(async resolve => {
          stream = await navigator.mediaDevices.getUserMedia({ audio: true })
          recorder = new MediaRecorder(stream)
          chunks = []
          recorder.ondataavailable = e => chunks.push(e.data)
          recorder.start()
          await sleep(time)
          recorder.onstop = async ()=>{
            blob = new Blob(chunks)
            text = await b2text(blob)
            resolve(text)
          }
          recorder.stop()
        })
        """
        display(Javascript(RECORD))
        audio_string = output.eval_js('record(10)')  # Record for 10 seconds
        audio_data = b64decode(audio_string.split(',')[1])

        # Recognize speech from audio data
        recognizer = sr.Recognizer()
        with io.BytesIO(audio_data) as source:
            audio = recognizer.record(source)
        try:
            return recognizer.recognize_google(audio)
        except sr.UnknownValueError:
            print("Sorry, could not understand the audio.")
            return None
        except sr.RequestError:
            print("Sorry, could not request results from Google Speech Recognition service.")
            return None

    elif choice == '3':
        # Upload MP3 file
        uploaded = files.upload()
        if uploaded:
            filename = list(uploaded.keys())[0]

            # Convert MP3 to WAV using pydub
            sound = AudioSegment.from_mp3(filename)
            sound.export(filename[:-3] + "wav", format="wav") # Export as WAV

            # Recognize speech from uploaded file (now in WAV format)
            recognizer = sr.Recognizer()
            with sr.AudioFile(filename[:-3] + "wav") as source: # Use the WAV file
                audio = recognizer.record(source)
            try:
                return recognizer.recognize_google(audio)
            except sr.UnknownValueError:
                print("Sorry, could not understand the audio.")
                return None
            except sr.RequestError:
                print("Sorry, could not request results from Google Speech Recognition service.")
                return None
        else:
            print("No file uploaded.")
            return None

# Analyze user input
while True:
    user_input = get_user_input()
    if user_input:
        user_score = analyze_user_input(user_input, model, tokenizer)
        print(f"\nUser Input Analysis Score: {user_score:.2f}")

        if user_score >= 0.7:
            print("\nHigh Risk: Immediate intervention recommended. Contact a counselor or helpline.")
        elif user_score >= 0.3:
            print("\nMedium Risk: Monitor closely and provide support resources.")
        else:
            print("\nLow Risk: No immediate action required, but stay vigilant.")
    else:
        print("No valid input provided.")

    continue_input = input("Do you want to analyze another input? (yes/no): ").strip().lower()
    if continue_input != 'yes':
        break

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/usr/local/lib/python3.11/dist-packages/transformers/optimization.py:591: FutureWarning: This implementation of AdamW is deprecated and will be removed in a future version. Use the PyTorch implementation torch.optim.AdamW instead, or set `no_deprecation_warning=True` to disable this warning
  warnings.warn(


Epoch 1/4


100%|██████████| 4/4 [00:04<00:00,  1.24s/it]


Train Loss: 0.626661092042923
Validation Accuracy: 85.7143
Epoch 2/4


100%|██████████| 4/4 [00:04<00:00,  1.24s/it]


...

100%|██████████| 4/4 [00:05<00:00,  1.28s/it]


Train Loss: 0.22116336226463318
Validation Accuracy: 100.0000
Enter '1' for text input, '2' to record audio, or '3' to upload an MP3 file: 1
Enter text to analyze: kill

User Input Analysis Score: 0.55

Medium Risk: Monitor closely and provide support resources.
Do you want to analyze another input? (yes/no): yes
Enter '1' for text input, '2' to record audio, or '3' to upload an MP3 file: 3


Saving WhatsApp Audio 2025-03-07 at 09.54.27_a6d5e02a.mp3 to WhatsApp Audio 2025-03-07 at 09.54.27_a6d5e02a (3).mp3

User Input Analysis Score: 0.42

Medium Risk: Monitor closely and provide support resources.
Do you want to analyze another input? (yes/no): yes
Enter '1' for text input, '2' to record audio, or '3' to upload an MP3 file: 3


Saving -2yygHLdpXc_out.wav to -2yygHLdpXc_out.wav
Sorry, could not understand the audio.
No valid input provided.
Do you want to analyze another input? (yes/no): yes
Enter '1' for text input, '2' to record audio, or '3' to upload an MP3 file: 3


Saving WhatsApp Audio 2025-03-07 at 09.54.27_a6d5e02a.mp3 to WhatsApp Audio 2025-03-07 at 09.54.27_a6d5e02a (4).mp3

User Input Analysis Score: 0.42

Medium Risk: Monitor closely and provide support resources.
Do you want to analyze another input? (yes/no): yes
Enter '1' for text input, '2' to record audio, or '3' to upload an MP3 file: 1
Enter text to analyze: They isolate me from my family and friends

User Input Analysis Score: 0.84

High Risk: Immediate intervention recommended. Contact a counselor or helpline.
Do you want to analyze another input? (yes/no): no
